# Limits of the group contrast (Moore et al. Batch_1, pancreatic cancer)

Everything here stays on DESeq2's own scale -- TMM-log2 residualized on each run's own RUVg W,
standardized on the batch-matched healthy controls. No normative model is used, so the
heterogeneity shown below cannot be an artefact of the normative Z.

The question is not whether the group contrast is significant, but whether the group mean it
reports describes the patients it was computed from.

In [ ]:
import sys

import pandas as pd

sys.path.insert(0, '..')
from viz_style import apply_style
apply_style()
import config
from analysis.plot_group_contrast import (centering_contrast, load, outlier_decomposition,
                                          plot_direction, plot_top_degs, representativeness)

OUT = config.EDA_GROUP_CONTRAST_DIR
FIG = config.EDA_GROUP_CONTRAST_FIG_DIR
FIG.mkdir(parents=True, exist_ok=True)

D = load()
print(f"cases={int(D['is_case'].sum())} HC={int(D['is_hc'].sum())} genes={len(D['genes'])}")

## 1. The group mean does not represent its own patients

`abs_mean_over_sd` is the group mean divided by the between-patient SD on the run's own DE genes.
`sign_concordance` asks only for direction: what fraction of patients sit on the side the group
mean claims. 0.5 is a coin flip.

In [ ]:
rep = representativeness(D, save_path=OUT / 'representativeness.csv')
rep.round(3)

## 2. Centering is not a free choice

Without centering both statistics stop asking about deviation and start reporting absolute
abundance. Centering on the healthy controls is the most conservative of the three options.

In [ ]:
centering_contrast(D, save_path=OUT / 'centering_contrast.csv').round(3)

## 3. The figures

In [ ]:
fig, direction_tbl = plot_direction(D, save_path=FIG / 'fig_group_direction.png')
direction_tbl.round(3)

In [ ]:
plot_top_degs(D, save_path=FIG / 'fig_group_top_degs_violin.png');

## 4. Is it a handful of outlier patients?

No. Extreme values are spread across nearly every patient and across a large share of the DE
genes, and dropping any single patient barely moves the group mean. The failure is pervasive
heterogeneity, not a few bad samples -- which is why trimming outliers would not rescue the
group contrast.

In [ ]:
outlier_decomposition(D, save_path=OUT / 'outlier_decomposition.csv').round(4)